# NanoMax PepAu-Forge — Real-Data Benchmark v0.4

**Purpose:** benchmark peptide-programmed gold nanocluster (AuNC) prediction using **only curated literature data** from `NanoMax_PepAuDB_v0.6.xlsx`.

This notebook deliberately does **not** generate synthetic labels and does **not** advertise a production accuracy score. It asks:

> *Does the currently available primary-literature dataset contain enough reproducible signal to outperform simple baselines under paper-wise and sequence-cluster-aware validation?*

### Scientific safeguards
- paper-wise grouped validation (`Source_ID` / `Paper_Group`);
- near-duplicate peptide clustering;
- target-specific data gates;
- median baseline before nonlinear models;
- ElasticNet, Random Forest, Gaussian Process, HistGradientBoosting and optional CatBoost;
- y-scrambling/permutation sanity checks;
- bootstrap confidence intervals;
- applicability-domain / out-of-distribution (OOD) scoring;
- no PTT model until direct peptide-AuNC PTT data pass a minimum evidence gate;
- no deployment unless an untouched external test set is available.

**Research prototype only — not a clinical or experimental substitute.**

In [ ]:
%pip -q install openpyxl catboost


In [ ]:
import os
import json
import math
import warnings

import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_VERSION = "PepAuDB_v0.6"
DATA_PATH = "/content/NanoMax_PepAuDB_v0.6.xlsx"
TRAINING_SHEET = "Training_View_v0.6"

# Diagnostic benchmark thresholds. Passing these permits exploratory benchmarking only.
MIN_N_DIAGNOSTIC = 6
MIN_PAPER_GROUPS = 3
MIN_SEQUENCE_CLUSTERS = 3

# Post-synthesis fraction labels may be outcome-dependent.
# Keep OFF for the inverse-design baseline.
ALLOW_POST_SYNTHESIS_FRACTION_FEATURES = False


## 1. Load the curated workbook

Upload `NanoMax_PepAuDB_v0.6.xlsx` into the Colab session if it is not already present.
The notebook reads the **curated training view**, not the entire literature table.

In [ ]:
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print(f"{DATA_PATH} not found. Upload NanoMax_PepAuDB_v0.6.xlsx now.")
        uploaded = files.upload()
        if "NanoMax_PepAuDB_v0.6.xlsx" in uploaded:
            DATA_PATH = "/content/NanoMax_PepAuDB_v0.6.xlsx"
        else:
            xlsx_files = [k for k in uploaded if k.lower().endswith(".xlsx")]
            if not xlsx_files:
                raise FileNotFoundError("No XLSX file was uploaded.")
            DATA_PATH = f"/content/{xlsx_files[0]}"
    except ImportError:
        raise FileNotFoundError(
            "Workbook not found. Set DATA_PATH to your local copy of NanoMax_PepAuDB_v0.6.xlsx."
        )

df = pd.read_excel(DATA_PATH, sheet_name=TRAINING_SHEET)
df.columns = [str(c).strip() for c in df.columns]
df = df[df["Record_ID"].notna()].copy()

print("Loaded:", DATA_PATH)
print("Rows in curated training view:", len(df))
display(df.head())


## 2. Provenance and target audit

The effective sample size is not the row count alone. Multiple rows can come from the same paper, peptide family, or synthesis/fractionation workflow. We therefore count total labeled rows, independent paper groups and independent sequence clusters.

In [ ]:
TARGETS = {
    "Emission_nm": "Emission peak (nm)",
    "PLQY_pct": "Photoluminescence quantum yield (%)",
    "Core_Size_nm": "Core size (nm)",
    "Au_Nuclearity": "Gold nuclearity (Au atom count)",
}

audit_rows = []
for target, label in TARGETS.items():
    sub = df[df[target].notna()].copy()
    audit_rows.append({
        "target": target,
        "label": label,
        "n_rows": len(sub),
        "paper_groups": sub["Paper_Group"].nunique(),
        "declared_sequence_clusters": sub["Sequence_Cluster"].nunique(),
    })

audit = pd.DataFrame(audit_rows)
display(audit)


## 3. Peptide feature engineering

The baseline descriptor layer encodes **sequence order as well as composition** and avoids the old arbitrary radius-of-gyration proxy.

It includes amino-acid fractions; Cys/His/Tyr/Trp/Met counts and positions; approximate charge, hydropathy and aromaticity; residue-spacing statistics; selected motifs; pH, temperature and time; and metal-composition flags.

Frozen ESM-2 embeddings should be added later as a separate ablation once the dataset contains more independent peptide families.

In [ ]:
AA = list("ACDEFGHIKLMNPQRSTVWY")

KYTE_DOOLITTLE = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}
AROMATIC = set("FYW")
COORDINATION = list("CHYWM")

def normalize_sequence(seq):
    """Convert common ligand notation to a model-safe amino-acid string where possible."""
    if pd.isna(seq):
        return ""
    s = str(seq).upper().strip()
    if s in {"Γ-ECG", "Γ‑ECG", "GSH", "GLUTATHIONE"}:
        return "ECG"
    s = s.replace("Γ-", "").replace("Γ", "")
    return "".join(ch for ch in s if ch in AA)

def spacing_stats(seq, residue):
    pos = [i for i, aa in enumerate(seq) if aa == residue]
    if len(pos) < 2:
        return (np.nan, np.nan, np.nan)
    d = np.diff(pos)
    return float(np.min(d)), float(np.mean(d)), float(np.max(d))

def position_stats(seq, residue):
    pos = [i for i, aa in enumerate(seq) if aa == residue]
    L = max(len(seq) - 1, 1)
    if not pos:
        return (np.nan, np.nan, np.nan)
    norm = np.asarray(pos, dtype=float) / L
    return float(norm[0]), float(np.mean(norm)), float(norm[-1])

def sequence_features(seq):
    s = normalize_sequence(seq)
    L = len(s)
    out = {"seq_length": L}

    for aa in AA:
        out[f"frac_{aa}"] = (s.count(aa) / L) if L else np.nan

    for aa in COORDINATION:
        out[f"count_{aa}"] = s.count(aa) if L else 0
        first, mean_pos, last = position_stats(s, aa)
        out[f"{aa}_first_norm"] = first
        out[f"{aa}_mean_norm"] = mean_pos
        out[f"{aa}_last_norm"] = last
        mn, avg, mx = spacing_stats(s, aa)
        out[f"{aa}_spacing_min"] = mn
        out[f"{aa}_spacing_mean"] = avg
        out[f"{aa}_spacing_max"] = mx

    if L:
        out["charge_proxy_pH7"] = (
            s.count("K") + s.count("R") + 0.1 * s.count("H")
            - s.count("D") - s.count("E")
        )
        out["aromaticity"] = sum(s.count(a) for a in AROMATIC) / L
        out["mean_hydropathy"] = float(np.mean([KYTE_DOOLITTLE[a] for a in s]))
        out["motif_CCY"] = s.count("CCY")
        out["motif_CXXC"] = sum(
            1 for i in range(max(0, L - 3))
            if s[i] == "C" and s[i + 3] == "C"
        )
        out["n_C_before_Y"] = sum(
            1 for i, a in enumerate(s) if a == "C" and "Y" in s[i + 1:]
        )
    else:
        out["charge_proxy_pH7"] = np.nan
        out["aromaticity"] = np.nan
        out["mean_hydropathy"] = np.nan
        out["motif_CCY"] = 0
        out["motif_CXXC"] = 0
        out["n_C_before_Y"] = 0

    return out

def build_feature_frame(frame):
    rows = []
    for _, r in frame.iterrows():
        f = sequence_features(r["Sequence"])
        f["pH"] = pd.to_numeric(r.get("pH"), errors="coerce")
        f["Temp_C"] = pd.to_numeric(r.get("Temp_C"), errors="coerce")
        f["Time_h"] = pd.to_numeric(r.get("Time_h"), errors="coerce")
        f["is_atomic_regime"] = float(str(r.get("Material_Regime")) == "atomically_precise_AuNC")
        f["is_AuCu"] = float(str(r.get("Metal_Composition")) == "AuCu")
        f["is_acetylated"] = float("acetyl" in str(r.get("Modification_State", "")).lower())
        f["ligand_count"] = pd.to_numeric(r.get("Ligand_Count"), errors="coerce")
        f["is_cyclic"] = float(str(r.get("Cyclization_State", "")).lower() == "cyclic")
        f["has_D_stereo"] = float("d-" in str(r.get("Chirality_State", "")).lower())
        f["is_photochemical"] = float("photochemical" in str(r.get("Protocol_Family", "")).lower())

        if ALLOW_POST_SYNTHESIS_FRACTION_FEATURES:
            state = str(r.get("Product_Process_State", "")).lower()
            f["is_supernatant_fraction"] = float("supernatant" in state)
            f["is_precipitate_fraction"] = float("precipitate" in state)

        rows.append(f)

    X = pd.DataFrame(rows, index=frame.index)
    X = X.replace([np.inf, -np.inf], np.nan)

    # Compact tiny-N descriptor set. Extended AA-fraction/position features are
    # calculated above but intentionally excluded from the first benchmark to
    # reduce dimensionality and overfitting risk.
    compact = [
        "seq_length",
        "count_C", "count_H", "count_Y", "count_W", "count_M",
        "C_first_norm", "C_mean_norm", "C_last_norm", "C_spacing_mean",
        "Y_first_norm", "Y_mean_norm", "Y_last_norm",
        "charge_proxy_pH7", "aromaticity", "mean_hydropathy",
        "motif_CCY", "n_C_before_Y",
        "pH", "Temp_C", "Time_h",
        "is_atomic_regime", "is_AuCu", "is_acetylated", "ligand_count", "is_cyclic", "has_D_stereo", "is_photochemical",
    ]
    if ALLOW_POST_SYNTHESIS_FRACTION_FEATURES:
        compact += ["is_supernatant_fraction", "is_precipitate_fraction"]

    return X[[c for c in compact if c in X.columns]]

X_all = build_feature_frame(df)
print("Numeric feature count:", X_all.shape[1])
display(X_all.head())


## 4. Independent near-duplicate sequence clustering

The workbook includes expert-assigned sequence families. This cell also calculates a transparent edit-distance clustering as an independent leakage check. A similarity threshold of `0.80` is used as a diagnostic rule, not a biological universal.

In [ ]:
def levenshtein_distance(a, b):
    a, b = normalize_sequence(a), normalize_sequence(b)
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        cur = [i]
        for j, cb in enumerate(b, start=1):
            cur.append(min(
                cur[-1] + 1,
                prev[j] + 1,
                prev[j - 1] + (ca != cb),
            ))
        prev = cur
    return prev[-1]

def sequence_similarity(a, b):
    aa, bb = normalize_sequence(a), normalize_sequence(b)
    denom = max(len(aa), len(bb), 1)
    return 1.0 - levenshtein_distance(aa, bb) / denom

def cluster_sequences(sequences, threshold=0.80):
    n = len(sequences)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    seqs = list(sequences)
    for i in range(n):
        for j in range(i + 1, n):
            if sequence_similarity(seqs[i], seqs[j]) >= threshold:
                union(i, j)

    roots = {}
    labels = []
    next_id = 1
    for i in range(n):
        r = find(i)
        if r not in roots:
            roots[r] = f"LEV_CLUSTER_{next_id:02d}"
            next_id += 1
        labels.append(roots[r])
    return labels

df["Levenshtein_Cluster"] = cluster_sequences(df["Sequence"].tolist(), threshold=0.80)

display(df[[
    "Record_ID", "Sequence", "Paper_Group",
    "Sequence_Cluster", "Levenshtein_Cluster"
]])


## 5. Target gates

A target enters the **diagnostic benchmark** only if it has enough labeled rows and at least three independent paper and sequence groups. Passing this gate does **not** mean production readiness.

In [ ]:
def target_gate(frame, target):
    sub = frame[frame[target].notna()].copy()
    info = {
        "target": target,
        "n": len(sub),
        "paper_groups": sub["Paper_Group"].nunique(),
        "sequence_clusters": sub["Levenshtein_Cluster"].nunique(),
    }
    info["diagnostic_gate"] = (
        info["n"] >= MIN_N_DIAGNOSTIC
        and info["paper_groups"] >= MIN_PAPER_GROUPS
        and info["sequence_clusters"] >= MIN_SEQUENCE_CLUSTERS
    )
    return info

gate_table = pd.DataFrame([target_gate(df, t) for t in TARGETS])
display(gate_table)


## 6. Candidate models and grouped out-of-fold evaluation

Model selection uses **paper-grouped out-of-fold predictions**. A median `DummyRegressor` is mandatory. If machine-learning models do not beat the dummy baseline, the scientifically correct conclusion is that PepAuDB is not yet predictive for that target.

In [ ]:
def make_models(n_features):
    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e3))
        + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-6, 1e3))
    )

    models = {
        "DummyMedian": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", DummyRegressor(strategy="median")),
        ]),
        "ElasticNet": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", ElasticNet(alpha=0.05, l1_ratio=0.20, max_iter=50000, random_state=RANDOM_SEED)),
        ]),
        "RandomForest": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=300,
                max_depth=3,
                min_samples_leaf=1,
                max_features="sqrt",
                random_state=RANDOM_SEED,
            )),
        ]),
        "HistGradientBoosting": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(
                max_iter=150,
                learning_rate=0.05,
                max_leaf_nodes=7,
                l2_regularization=1.0,
                random_state=RANDOM_SEED,
            )),
        ]),
        "GaussianProcess": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", GaussianProcessRegressor(
                kernel=kernel,
                normalize_y=True,
                n_restarts_optimizer=2,
                random_state=RANDOM_SEED,
            )),
        ]),
    }

    try:
        from catboost import CatBoostRegressor
        models["CatBoost"] = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", CatBoostRegressor(
                iterations=250,
                depth=3,
                learning_rate=0.03,
                loss_function="MAE",
                verbose=False,
                random_seed=RANDOM_SEED,
                allow_writing_files=False,
            )),
        ])
    except Exception as e:
        print("CatBoost unavailable; continuing without it:", e)

    return models

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rho = spearmanr(y_true, y_pred).statistic
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": math.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan,
        "Spearman": float(rho) if np.isfinite(rho) else np.nan,
    }

def grouped_oof(model, X, y, groups):
    groups = np.asarray(groups)
    unique_groups = np.unique(groups)
    if len(unique_groups) < 2:
        raise ValueError("At least two groups are required.")
    n_splits = min(5, len(unique_groups))
    cv = GroupKFold(n_splits=n_splits)

    oof = np.full(len(y), np.nan, dtype=float)
    fold_id = np.full(len(y), -1, dtype=int)

    for fold, (tr, te) in enumerate(cv.split(X, y, groups=groups), start=1):
        m = clone(model)
        m.fit(X.iloc[tr], y.iloc[tr])
        oof[te] = np.asarray(m.predict(X.iloc[te]), dtype=float).reshape(-1)
        fold_id[te] = fold

    return oof, fold_id

def bootstrap_metric_ci(y, pred, metric="MAE", n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    y = np.asarray(y, float)
    pred = np.asarray(pred, float)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        yy, pp = y[idx], pred[idx]
        if metric == "MAE":
            vals.append(mean_absolute_error(yy, pp))
        elif metric == "RMSE":
            vals.append(math.sqrt(mean_squared_error(yy, pp)))
        else:
            raise ValueError(metric)
    return tuple(np.percentile(vals, [2.5, 97.5]))


## 7. Run the first real-data benchmark

The first pass uses **paper group** as the primary cross-validation unit. A separate sequence-cluster-held-out stress test is also run where feasible.

In [ ]:
all_results = []
oof_store = {}

for target, label in TARGETS.items():
    gate = target_gate(df, target)
    print("\n" + "=" * 88)
    print(label, gate)

    if not gate["diagnostic_gate"]:
        print("STOP: target does not pass the diagnostic benchmark gate.")
        continue

    sub = df[df[target].notna()].copy().reset_index(drop=True)
    X = build_feature_frame(sub).reset_index(drop=True)
    y = pd.to_numeric(sub[target], errors="coerce").astype(float).reset_index(drop=True)
    models = make_models(X.shape[1])

    validation_groups = {
        "paper_grouped": sub["Paper_Group"].values,
        "lab_grouped": sub["Lab_Group"].values if "Lab_Group" in sub.columns else sub["Paper_Group"].values,
    }

    for validation_name, groups in validation_groups.items():
        if len(np.unique(groups)) < 3:
            continue

        for model_name, model in models.items():
            pred, folds = grouped_oof(model, X, y, groups)
            met = regression_metrics(y, pred)
            lo, hi = bootstrap_metric_ci(y, pred, "MAE", n_boot=1000, seed=RANDOM_SEED)

            all_results.append({
                "target": target,
                "label": label,
                "model": model_name,
                "validation": validation_name,
                "n": len(sub),
                "paper_groups": sub["Paper_Group"].nunique(),
                "lab_groups": sub["Lab_Group"].nunique() if "Lab_Group" in sub.columns else sub["Paper_Group"].nunique(),
                "sequence_clusters": sub["Levenshtein_Cluster"].nunique(),
                **met,
                "MAE_CI_low": lo,
                "MAE_CI_high": hi,
            })
            oof_store[(target, model_name, validation_name)] = {
                "y": y.to_numpy(),
                "pred": pred,
                "record_id": sub["Record_ID"].to_numpy(),
                "groups": np.asarray(groups),
                "fold": folds,
            }

    # Sequence-cluster-held-out stress test remains a third, stricter view.
    if sub["Levenshtein_Cluster"].nunique() >= MIN_SEQUENCE_CLUSTERS:
        for model_name, model in models.items():
            pred, folds = grouped_oof(model, X, y, sub["Levenshtein_Cluster"].values)
            met = regression_metrics(y, pred)
            lo, hi = bootstrap_metric_ci(y, pred, "MAE", n_boot=1000, seed=RANDOM_SEED)
            all_results.append({
                "target": target,
                "label": label,
                "model": model_name,
                "validation": "sequence_cluster_grouped",
                "n": len(sub),
                "paper_groups": sub["Paper_Group"].nunique(),
                "lab_groups": sub["Lab_Group"].nunique() if "Lab_Group" in sub.columns else sub["Paper_Group"].nunique(),
                "sequence_clusters": sub["Levenshtein_Cluster"].nunique(),
                **met,
                "MAE_CI_low": lo,
                "MAE_CI_high": hi,
            })

results = pd.DataFrame(all_results)

if results.empty:
    print("No target currently passes the diagnostic benchmark gate.")
else:
    display(results.sort_values(["target", "validation", "MAE"]))


## 8. Baseline comparison

This reports improvement over the median baseline. With the current sample size, any improvement remains a **diagnostic signal**, not a stable accuracy estimate.

In [ ]:
baseline_summary = pd.DataFrame()

if not results.empty:
    paper_res = results[results["validation"] == "paper_grouped"].copy()
    summaries = []

    for target in paper_res["target"].unique():
        rr = paper_res[paper_res["target"] == target].sort_values("MAE")
        dummy = rr[rr["model"] == "DummyMedian"]
        learned = rr[rr["model"] != "DummyMedian"]

        if dummy.empty or learned.empty:
            continue

        d_mae = float(dummy.iloc[0]["MAE"])
        best = learned.iloc[0]
        improvement = 100.0 * (d_mae - float(best["MAE"])) / max(d_mae, 1e-12)

        summaries.append({
            "target": target,
            "best_model": best["model"],
            "best_MAE": best["MAE"],
            "dummy_MAE": d_mae,
            "MAE_improvement_vs_dummy_pct": improvement,
            "interpretation": (
                "PROMISING_DIAGNOSTIC_SIGNAL"
                if improvement > 10
                else "WEAK_OR_UNCERTAIN_SIGNAL"
            ),
        })

    baseline_summary = pd.DataFrame(summaries)
    display(baseline_summary)


## 9. Y-scrambling sanity check

If performance remains similar after labels are shuffled, the apparent signal is not trustworthy.

In [ ]:
def y_scramble_test(frame, target, model, n_perm=50):
    sub = frame[frame[target].notna()].copy().reset_index(drop=True)
    X = build_feature_frame(sub).reset_index(drop=True)
    y = pd.to_numeric(sub[target], errors="coerce").astype(float).reset_index(drop=True)
    groups = sub["Paper_Group"].values

    real_pred, _ = grouped_oof(model, X, y, groups)
    real_mae = mean_absolute_error(y, real_pred)

    rng = np.random.default_rng(RANDOM_SEED)
    null_mae = []
    for _ in range(n_perm):
        y_perm = pd.Series(rng.permutation(y.to_numpy()))
        pred, _ = grouped_oof(model, X, y_perm, groups)
        null_mae.append(mean_absolute_error(y_perm, pred))

    null_mae = np.asarray(null_mae)
    p_like = (1 + np.sum(null_mae <= real_mae)) / (1 + len(null_mae))
    return {
        "real_MAE": real_mae,
        "null_MAE_median": float(np.median(null_mae)),
        "null_MAE_2.5pct": float(np.percentile(null_mae, 2.5)),
        "null_MAE_97.5pct": float(np.percentile(null_mae, 97.5)),
        "permutation_p_like": float(p_like),
    }

scramble_rows = []
if not results.empty:
    paper_res = results[results["validation"] == "paper_grouped"]
    for target in paper_res["target"].unique():
        rr = paper_res[
            (paper_res["target"] == target) &
            (paper_res["model"] != "DummyMedian")
        ]
        if rr.empty:
            continue

        best_name = rr.sort_values("MAE").iloc[0]["model"]
        sub = df[df[target].notna()].copy().reset_index(drop=True)
        X = build_feature_frame(sub)
        models = make_models(X.shape[1])

        if best_name in models:
            s = y_scramble_test(df, target, models[best_name], n_perm=50)
            scramble_rows.append({"target": target, "model": best_name, **s})

scramble_results = pd.DataFrame(scramble_rows)
display(scramble_results)


## 10. Applicability domain (OOD) and nearest literature analogue

A future candidate should receive an OOD warning and nearest-literature records alongside any prediction.

In [ ]:
def fit_applicability_domain(reference_frame):
    X = build_feature_frame(reference_frame).copy()
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    Xi = imputer.fit_transform(X)
    Xs = scaler.fit_transform(Xi)

    nn = NearestNeighbors(n_neighbors=min(2, len(reference_frame)))
    nn.fit(Xs)
    distances, _ = nn.kneighbors(Xs)

    train_nn = distances[:, 1] if distances.shape[1] == 2 else distances[:, 0]
    threshold95 = float(np.percentile(train_nn, 95)) if len(train_nn) else np.nan

    return {
        "feature_columns": list(X.columns),
        "imputer": imputer,
        "scaler": scaler,
        "nn": nn,
        "threshold95": threshold95,
        "reference_records": reference_frame["Record_ID"].to_numpy(),
    }

ad = fit_applicability_domain(df)
print("95th-percentile within-literature nearest-neighbor distance:", ad["threshold95"])

def candidate_feature_row(sequence, pH=np.nan, Temp_C=np.nan, Time_h=np.nan,
                          material_regime="ultrasmall_AuNC_unknown_nuclearity",
                          metal_composition="pure_Au"):
    row = pd.DataFrame([{
        "Sequence": sequence,
        "pH": pH,
        "Temp_C": Temp_C,
        "Time_h": Time_h,
        "Material_Regime": material_regime,
        "Metal_Composition": metal_composition,
        "Product_Process_State": "unknown_pre_synthesis",
    }])
    return build_feature_frame(row)

def applicability_for_candidate(sequence, pH=np.nan, Temp_C=np.nan, Time_h=np.nan):
    Xc = candidate_feature_row(sequence, pH, Temp_C, Time_h)
    Xc = Xc.reindex(columns=ad["feature_columns"])
    Xci = ad["imputer"].transform(Xc)
    Xcs = ad["scaler"].transform(Xci)

    dist, idx = ad["nn"].kneighbors(Xcs, n_neighbors=min(3, len(df)))
    nearest = [
        {"Record_ID": ad["reference_records"][i], "distance": float(d)}
        for d, i in zip(dist[0], idx[0])
    ]
    d1 = float(dist[0][0])
    return {
        "nearest_distance": d1,
        "ood_flag": bool(np.isfinite(ad["threshold95"]) and d1 > ad["threshold95"]),
        "nearest_literature_records": nearest,
    }

example_ad = applicability_for_candidate("CCYLRRASLG", pH=9, Temp_C=25, Time_h=13)
example_ad


## 11. Prediction uncertainty

A bootstrap ensemble gives a transparent first uncertainty signal. It is not a substitute for calibration on larger independent data.

In [ ]:
def bootstrap_prediction_dispersion(model, X, y, candidate_X, n_models=200, seed=42):
    rng = np.random.default_rng(seed)
    preds = []

    for _ in range(n_models):
        idx = rng.integers(0, len(y), len(y))
        m = clone(model)
        m.fit(X.iloc[idx], y.iloc[idx])
        preds.append(float(np.asarray(m.predict(candidate_X)).reshape(-1)[0]))

    preds = np.asarray(preds)
    return {
        "median": float(np.median(preds)),
        "p2.5": float(np.percentile(preds, 2.5)),
        "p97.5": float(np.percentile(preds, 97.5)),
        "sd": float(np.std(preds, ddof=1)),
    }


## 12. PTT hard gate

PTT remains a primary project objective, but PepAuDB v0.3 still lacks enough **direct peptide-templated AuNC sequence+synthesis→PTT** rows for a defensible learned regression.

The PTT layer therefore remains a benchmark/reference layer until direct records include peptide sequence, material regime/nuclearity or size, synthesis context, laser wavelength, power density, concentration, and a quantitative PCE or ΔT/heating endpoint across independent papers.

In [ ]:
PTT_DIRECT_MIN_ROWS = 12
PTT_DIRECT_MIN_PAPERS = 4
PTT_DIRECT_MIN_SEQUENCE_FAMILIES = 4

ptt_gate = {
    "status": "STOP",
    "reason": (
        "PepAuDB v0.3 does not yet contain enough direct peptide-templated "
        "AuNC PTT rows for a defensible learned model."
    ),
    "required_min_rows_for_first_diagnostic": PTT_DIRECT_MIN_ROWS,
    "required_min_papers": PTT_DIRECT_MIN_PAPERS,
    "required_min_sequence_families": PTT_DIRECT_MIN_SEQUENCE_FAMILIES,
    "production_claim": False,
}
ptt_gate


## v0.6 extension — laboratory-aware stress testing

Paper-wise splitting is necessary but can still be optimistic when two papers come from the same laboratory and reuse a closely related synthesis platform.

PepAuDB v0.6 therefore introduces `Lab_Group`. In particular, the Au16(RGDC)14 and Au22(KCK)16 photochemical studies are grouped into one **STAMPLECOSKIE_PHOTOCHEM** laboratory family during the lab-aware stress test.

A target should not receive a production claim merely because paper-grouped performance is better than lab-grouped performance.

In [ ]:
if "Lab_Group" in df.columns:
    print("Independent paper groups:", df["Paper_Group"].nunique())
    print("Independent lab groups:", df["Lab_Group"].nunique())
    display(df[["Record_ID","Source_ID","Sequence","Paper_Group","Lab_Group"]].tail(10))
else:
    print("Lab_Group column not found; update to PepAuDB v0.6 training view.")

## 13. Model card and result export

The model card records what was actually tested and deliberately sets `production_ready = false`.

## v0.4 extension — pairwise photoluminescence ranking evidence

PepAuDB v0.4 contains verified **relative** photoluminescence comparisons from the 2018 JACS peptide panel.
These are scientifically useful even when absolute QY is unavailable.

The correct use is a separate pairwise-ranking task, not conversion into fabricated absolute labels.

In [ ]:
try:
    relative_pl = pd.read_excel(DATA_PATH, sheet_name="Relative_PL_Panel_v0.6")
    relative_pl = relative_pl[relative_pl["Panel_ID"].notna()].copy()
    display(relative_pl[[
        "Panel_ID","Sequence_A","Sequence_B","Comparison_Type",
        "Observed_Relationship","Fold_Change","Training_Use"
    ]])
    print("Verified relative-ranking comparisons:", len(relative_pl))
except Exception as e:
    print("Relative PL panel could not be loaded:", e)

## v0.4 extension — explicit formation negative

The 2017 Pep II experiment is a rare matched failure/control: under the same reported conditions,
the shorter peptide produced almost no fluorescence and many gold nanoparticles instead of the
successful AuNC suprastructure. This is valuable, but **one negative is not enough** to train a formation classifier.

In [ ]:
try:
    formation_controls = pd.read_excel(DATA_PATH, sheet_name="Formation_Controls_v0.6")
    formation_controls = formation_controls[formation_controls["Control_ID"].notna()].copy()
    display(formation_controls)
    print("Explicit verified formation negatives:", len(formation_controls))
    if len(formation_controls) < 5:
        print("STOP: formation classifier remains disabled.")
except Exception as e:
    print("Formation-control sheet could not be loaded:", e)

In [ ]:
model_card = {
    "project": "NanoMax PepAu-Forge",
    "benchmark_version": "Real-Data Benchmark v0.4",
    "data_version": DATA_VERSION,
    "dataset_path": os.path.basename(DATA_PATH),
    "training_sheet": TRAINING_SHEET,
    "random_seed": RANDOM_SEED,
    "synthetic_labels_used": False,
    "primary_validation": "paper-grouped out-of-fold cross-validation",
    "lab_aware_stress_test": "Lab_Group grouped out-of-fold validation, combining known same-platform papers",
    "secondary_validation": "sequence-cluster-grouped stress test when feasible",
    "post_synthesis_fraction_features_enabled": ALLOW_POST_SYNTHESIS_FRACTION_FEATURES,
    "diagnostic_thresholds": {
        "min_labeled_rows": MIN_N_DIAGNOSTIC,
        "min_paper_groups": MIN_PAPER_GROUPS,
        "min_sequence_clusters": MIN_SEQUENCE_CLUSTERS,
    },
    "target_gates": [target_gate(df, t) for t in TARGETS],
    "ptt_gate": ptt_gate,
    "external_validation_completed": False,
    "production_ready": False,
    "permitted_claim": (
        "Exploratory retrospective benchmark only. "
        "No experimental truth, clinical validity, or production generalization claim."
    ),
}

if not results.empty:
    model_card["benchmark_results"] = json.loads(
        results.replace({np.nan: None}).to_json(orient="records")
    )

with open("/content/NanoMax_PepAu_model_card_v0.4.json", "w") as f:
    json.dump(model_card, f, indent=2)

if not results.empty:
    results.to_csv("/content/NanoMax_PepAu_benchmark_results_v0.4.csv", index=False)

with open("/content/NanoMax_PepAu_model_card_v0.4.json") as f:
    print(f.read())


## v0.5 milestone — atomic nuclearity benchmark is now diagnostically allowed

PepAuDB v0.5 reaches the unchanged diagnostic minimum for the **Au nuclearity** target:
six labeled records across at least three independent paper and sequence groups.

This only permits a first diagnostic benchmark. It does **not** imply the target is predictable.
If learned models fail to beat the median baseline, the notebook must report that failure rather than relax the gate or tune toward a favorable number.

In [ ]:
nuclearity_gate = target_gate(df, "Au_Nuclearity")
print(nuclearity_gate)
if nuclearity_gate["diagnostic_gate"]:
    print("GO for diagnostic nuclearity benchmark only; production remains gated.")
else:
    print("STOP: nuclearity diagnostic gate not met.")

## 14. Deployment decision

### Current decision
- **Database / evidence Streamlit:** GO.
- **Leakage-safe retrospective benchmark:** GO, diagnostic only.
- **Emission / size / QY predictor:** exploratory only when its target gate passes.
- **Atomic nuclearity predictor:** STOP for production; too few independent atomic peptide families.
- **Formation classifier:** STOP until explicit negative/failed syntheses are curated.
- **PTT predictor:** STOP until direct peptide-AuNC PTT rows are acquired.
- **“Highest accuracy” badge:** not scientifically permissible yet.
- **Wet-lab candidate selection:** later, after retrospective signal + uncertainty + OOD checks; prospective synthesis remains the true validation.

### Next evidence additions
1. Extract the 2022 seven-sequence CCY/KFFAAK panel at sequence×condition level.
2. Recover exact Liu-2020 blue/red synthesis conditions.
3. Acquire the Wen-2013 short-peptide panel and verify any Au8/QY mapping from the primary source.
4. Complete Au25Sv9 synthesis details and CMMMMM/CYYYYY SI.
5. Expand direct peptide-AuNC PTT records.
6. Re-run this notebook without relaxing validation rules.
7. Freeze future independent papers/labs as an untouched external test set.